In [ ]:
import glob
import numpy as np
import pandas as pd

files = sorted(glob.glob("../data/AVA/ava_activespeaker_test_v1.0/*-activespeaker.csv"))
cols = ["video", "t", "x1", "y1", "x2", "y2", "label", "entity"]
raw = pd.concat([pd.read_csv(f, header=None, names=cols) for f in files], ignore_index=True)

print("files:", len(files), "| rows:", len(raw), "| videos:", raw["video"].nunique())
print("labels:")
print(raw["label"].value_counts().to_string())
print("entities (all):", raw.groupby(["video", "entity"]).ngroups)

raw = raw.sort_values(["video", "entity", "t"]).reset_index(drop=True)
dt = raw.groupby(["video", "entity"])["t"].diff()
STEP = float(dt.median())
print("median frame step (s):", round(STEP, 4), "| 99th pct step:", round(float(dt.quantile(0.99)), 4))

MAXGAP = 0.2    # seconds; bridges single missing or flickering frames
MIN_DUR = 0.3   # seconds; shorter runs are dropped

sp = raw[raw["label"] == "SPEAKING_AUDIBLE"].copy()
sp["gap"] = sp.groupby(["video", "entity"])["t"].diff()
sp["run"] = ((sp["gap"].isna()) | (sp["gap"] > MAXGAP)).cumsum()
iv = (sp.groupby("run")
        .agg(video=("video", "first"), entity=("entity", "first"),
             start=("t", "min"), end=("t", "max"))
        .reset_index(drop=True))
iv["end"] = iv["end"] + STEP
iv["dur"] = iv["end"] - iv["start"]
iv = iv[iv["dur"] >= MIN_DUR].copy()

iv["corpus"] = "AVA"
iv["meeting"] = iv["video"]
iv["series"] = iv["video"]
iv["speaker"] = iv["video"] + ":" + iv["entity"]
iv["role"] = np.nan
iv["gender"] = np.nan
iv["gender_source"] = "none"
iv["n_words"] = np.nan
iv = iv[["corpus", "meeting", "series", "speaker", "role", "gender", "gender_source",
         "start", "end", "dur", "n_words"]]
iv.to_csv("../outputs/ava_intervals.csv", index=False)

print("intervals:", len(iv), "| total speaking hours:", round(iv["dur"].sum() / 3600, 2))
print("median interval (s):", round(float(iv["dur"].median()), 2))
per = iv.groupby("meeting").agg(entities=("speaker", "nunique"), intervals=("dur", "size"),
                                 speaking_s=("dur", "sum"))
print(per.describe().round(1).to_string())
print("videos with no speaking intervals:", raw["video"].nunique() - per.shape[0])

In [ ]:
import pandas as pd

iv = pd.read_csv("../outputs/ava_intervals.csv")
ent = iv["speaker"].str.split(":", n=1).str[1]
iv["window"] = ent.str.split(":").str[0]
iv["meeting"] = iv["window"]
iv["speaker"] = ent
iv.to_csv("../outputs/ava_intervals.csv", index=False)

g = iv.groupby("meeting").agg(entities=("speaker", "nunique"), intervals=("dur", "size"),
                              speaking_s=("dur", "sum"))
print("windows:", len(g), "| videos:", iv["series"].nunique())
print(g.describe().round(1).to_string())
print("windows with >= 20 intervals:", int((g["intervals"] >= 20).sum()),
      "| with >= 2 entities:", int((g["entities"] >= 2).sum()),
      "| both:", int(((g["intervals"] >= 20) & (g["entities"] >= 2)).sum()))

In [ ]:
import glob
import numpy as np
import pandas as pd

for f in sorted(glob.glob("../outputs/*interval*.csv")):
    d = pd.read_csv(f)
    print("FILE:", f, "| shape:", d.shape)
    print("columns:", list(d.columns))
    need = ["speaker", "gender", "start", "end"]
    miss = [c for c in need if c not in d.columns]
    if miss:
        print("MISSING columns:", miss)
        continue
    print("nulls:", d[need].isna().sum().to_dict())
    print("end <= start:", int((d["end"] <= d["start"]).sum()), "| start < 0:", int((d["start"] < 0).sum()))
    print("gender counts:", d["gender"].value_counts(dropna=False).to_dict())
    key = "meeting" if "meeting" in d.columns else None
    if key:
        d = d.sort_values([key, "speaker", "start"])
        prev_end = d.groupby([key, "speaker"])["end"].shift()
        ov = int((d["start"] < prev_end - 1e-6).sum())
        per = d.groupby(key).size()
        print("clips:", d[key].nunique(), "| speakers:", d["speaker"].nunique(),
              "| rows per clip min/median/max:", int(per.min()), int(per.median()), int(per.max()))
        print("same-speaker overlaps:", ov)
    print("duration s (median / max):", round(float((d["end"] - d["start"]).median()), 2),
          round(float((d["end"] - d["start"]).max()), 1))
    print("------")

In [ ]:
import pandas as pd

def overlap_check(d, keys):
    d = d.sort_values(keys + ["speaker", "start"])
    prev_end = d.groupby(keys + ["speaker"])["end"].shift()
    ov = d[d["start"] < prev_end - 1e-6]
    return len(ov), ov

ami = pd.read_csv("../outputs/ami_intervals.csv")
n, ov = overlap_check(ami, ["unit_id"])
print("AMI clips:", ami["unit_id"].nunique(), "| speakers:", ami["speaker"].nunique(),
      "| same-speaker overlaps:", n)
print("AMI intervals longer than 60 s:", int((ami["dur"] > 60).sum()),
      "| share of speaking time in them:", round(float(ami.loc[ami["dur"] > 60, "dur"].sum() / ami["dur"].sum()), 3))
print(ami.nlargest(5, "dur")[["unit_id", "speaker", "gender", "start", "end", "dur", "n_words"]].to_string(index=False))

meld = pd.read_csv("../outputs/meld_intervals_words.csv")
n2, ov2 = overlap_check(meld, ["season", "episode", "dialogue_id"])
print("MELD dialogues:", meld.groupby(["season", "episode", "dialogue_id"]).ngroups,
      "| speakers:", meld["speaker"].nunique(), "| same-speaker overlaps:", n2)
print("MELD intervals longer than 15 s:", int((meld["dur"] > 15).sum()))
print(meld.nlargest(5, "dur")[["season", "episode", "dialogue_id", "speaker", "dur", "n_words"]].to_string(index=False))

In [ ]:
import pandas as pd

ami = pd.read_csv("../outputs/ami_intervals.csv")
meld = pd.read_csv("../outputs/meld_intervals_words.csv")
ava = pd.read_csv("../outputs/ava_intervals.csv")

a = pd.DataFrame({"corpus": "AMI", "clip_id": "AMI:" + ami["unit_id"].astype(str),
                  "person_id": "AMI:" + ami["speaker"].astype(str), "series": ami["series"],
                  "component": ami["component"], "gender": ami["gender"],
                  "gender_source": ami["gender_source"], "start": ami["start"], "end": ami["end"],
                  "dur": ami["dur"], "n_words": ami["n_words"]})

s = meld["season"].astype(int).astype(str).str.zfill(2)
e = meld["episode"].astype(int).astype(str).str.zfill(2)
m = pd.DataFrame({"corpus": "MELD", "clip_id": "MELD:S" + s + "E" + e,
                  "person_id": "MELD:" + meld["speaker"].astype(str), "series": "MELD_S" + s,
                  "component": "MELD_ALL", "gender": meld["gender"], "gender_source": "manual_map",
                  "start": meld["start"], "end": meld["end"], "dur": meld["dur"],
                  "n_words": meld["n_words"]})

v = pd.DataFrame({"corpus": "AVA", "clip_id": "AVA:" + ava["meeting"].astype(str),
                  "person_id": "AVA:" + ava["speaker"].astype(str), "series": ava["series"],
                  "component": "AVA_" + ava["series"].astype(str), "gender": pd.NA,
                  "gender_source": "none", "start": ava["start"], "end": ava["end"],
                  "dur": ava["dur"], "n_words": pd.NA})

allv = pd.concat([a, m, v], ignore_index=True)
allv.to_csv("../outputs/shared/intervals_all.csv", index=False)
print(allv.groupby("corpus").agg(rows=("dur", "size"), clips=("clip_id", "nunique"),
                                 persons=("person_id", "nunique")).to_string())
print("columns:", list(allv.columns))